# MultiStream — análisis de salidas sin reentrenamiento

Este cuaderno **no entrena ni carga modelos**. Lee exclusivamente las salidas ya generadas en:

```text
/kaggle/input/notebooks/alejandragomezr/multistream
```

Calcula las métricas descriptivas por ventanas cuando las predicciones están disponibles y realiza la evaluación por participante solicitada por el revisor: accuracy, sensibilidad, especificidad, precisión, F1-score y ROC-AUC, con IC del 95 % mediante 5,000 remuestreos estratificados de participantes. También exporta las predicciones por semilla y el consenso necesario para McNemar y las comparaciones pareadas.

In [1]:
from pathlib import Path
import json
import shutil

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


INPUT_ROOT = Path("/kaggle/input/notebooks/alejandragomezr/multistream")
OUTPUT_ROOT = Path("/kaggle/working/MultiStream_analysis_only")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def find_export(root, exact_name, fallback_name=None, required=True):
    """Locate an exact consolidated export, with an optional fold-file fallback."""
    exact_matches = sorted(root.rglob(exact_name), key=lambda p: (len(p.parts), str(p)))
    if exact_matches:
        if len(exact_matches) > 1:
            print(f"Aviso: se encontraron {len(exact_matches)} copias de {exact_name}.")
            print("Se utilizará:", exact_matches[0])
        return exact_matches[0]

    if fallback_name is not None:
        fallback_matches = sorted(root.rglob(fallback_name))
        if fallback_matches:
            print(
                f"No se encontró {exact_name}; se consolidarán "
                f"{len(fallback_matches)} archivos {fallback_name}."
            )
            return fallback_matches

    if required:
        available = sorted(p.name for p in root.rglob("*.csv"))
        raise FileNotFoundError(
            f"No se encontró {exact_name} dentro de:\n{root}\n"
            f"CSV disponibles: {available[:50]}"
        )
    return None


if not INPUT_ROOT.exists():
    raise FileNotFoundError(
        "La salida del notebook no está adjunta en la ruta esperada:\n"
        f"{INPUT_ROOT}"
    )

subject_export = find_export(
    INPUT_ROOT,
    "MultiStream_subject_summary_by_seed.csv",
    fallback_name="subject_summary.csv",
    required=True,
)

if isinstance(subject_export, list):
    subject_frames = [pd.read_csv(path) for path in subject_export]
    subject_input = pd.concat(subject_frames, ignore_index=True)
    SUBJECT_SUMMARY_PATH = OUTPUT_ROOT / "MultiStream_subject_summary_by_seed.csv"
    subject_input.to_csv(SUBJECT_SUMMARY_PATH, index=False)
else:
    SUBJECT_SUMMARY_PATH = subject_export

WINDOW_PREDICTIONS_PATH = find_export(
    INPUT_ROOT,
    "MultiStream_all_window_predictions.csv",
    fallback_name="test_window_predictions.csv",
    required=False,
)

if isinstance(WINDOW_PREDICTIONS_PATH, list):
    window_frames = [pd.read_csv(path) for path in WINDOW_PREDICTIONS_PATH]
    window_input = pd.concat(window_frames, ignore_index=True)
    WINDOW_PREDICTIONS_PATH = OUTPUT_ROOT / "MultiStream_all_window_predictions.csv"
    window_input.to_csv(WINDOW_PREDICTIONS_PATH, index=False)

print("INPUT_ROOT:", INPUT_ROOT)
print("Resumen por participante y seed:", SUBJECT_SUMMARY_PATH)
print("Predicciones por ventana:", WINDOW_PREDICTIONS_PATH)
print("Resultados nuevos:", OUTPUT_ROOT)


INPUT_ROOT: /kaggle/input/notebooks/alejandragomezr/multistream
Resumen por participante y seed: /kaggle/input/notebooks/alejandragomezr/multistream/resultados_multistream_tdah_ARTICULO_120subjects/MultiStream_subject_summary_by_seed.csv
Predicciones por ventana: /kaggle/input/notebooks/alejandragomezr/multistream/resultados_multistream_tdah_ARTICULO_120subjects/MultiStream_all_window_predictions.csv
Resultados nuevos: /kaggle/working/MultiStream_analysis_only


## Métricas descriptivas por ventanas

In [2]:
def calculate_window_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    y_pred = (y_prob >= threshold).astype(np.int64)

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "sensitivity": float(tp / (tp + fn)) if (tp + fn) else np.nan,
        "specificity": float(tn / (tn + fp)) if (tn + fp) else np.nan,
        "precision": float(
            precision_score(y_true, y_pred, zero_division=0)
        ),
        "f1_score": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


if WINDOW_PREDICTIONS_PATH is not None:
    window_predictions = pd.read_csv(WINDOW_PREDICTIONS_PATH)
    probability_column = next(
        (
            column
            for column in ("prob_adhd", "y_prob_adhd", "subject_prob_adhd")
            if column in window_predictions.columns
        ),
        None,
    )
    label_column = next(
        (column for column in ("y_true", "label") if column in window_predictions.columns),
        None,
    )
    required = {"seed", "subject_id"}
    missing = required - set(window_predictions.columns)
    if missing or probability_column is None or label_column is None:
        raise KeyError(
            "Las predicciones por ventana no contienen las columnas necesarias. "
            f"Columnas: {list(window_predictions.columns)}"
        )

    window_rows = []
    for seed, seed_frame in window_predictions.groupby("seed", sort=True):
        metrics = calculate_window_metrics(
            seed_frame[label_column].to_numpy(),
            seed_frame[probability_column].to_numpy(),
        )
        window_rows.append(
            {
                "model": "MultiStream",
                "seed": int(seed),
                "n_windows": int(len(seed_frame)),
                "n_subjects": int(seed_frame["subject_id"].nunique()),
                **metrics,
            }
        )

    window_metrics_by_seed = pd.DataFrame(window_rows)
    metric_names = (
        "accuracy", "sensitivity", "specificity",
        "precision", "f1_score", "roc_auc",
    )
    window_summary_rows = []
    for metric in metric_names:
        values = window_metrics_by_seed[metric].to_numpy(dtype=float)
        window_summary_rows.append(
            {
                "metric": metric,
                "mean": float(np.nanmean(values)),
                "std_sample": float(np.nanstd(values, ddof=1)),
                "n_seeds": int(len(values)),
            }
        )
    window_metrics_summary = pd.DataFrame(window_summary_rows)

    window_metrics_by_seed.to_csv(
        OUTPUT_ROOT / "MultiStream_window_level_metrics_by_seed.csv", index=False
    )
    window_metrics_summary.to_csv(
        OUTPUT_ROOT / "MultiStream_window_level_metrics_summary.csv", index=False
    )

    print("\nMétricas descriptivas por ventanas (media ± DE entre seeds):")
    print(window_metrics_summary.to_string(index=False))
else:
    print(
        "No se encontraron predicciones por ventana. "
        "Se continuará con el análisis por participante."
    )



Métricas descriptivas por ventanas (media ± DE entre seeds):
     metric     mean  std_sample  n_seeds
   accuracy 0.586278    0.002922       10
sensitivity 0.861414    0.009387       10
specificity 0.243506    0.014280       10
  precision 0.586564    0.002521       10
   f1_score 0.697875    0.002526       10
    roc_auc 0.552511    0.003872       10


## Resultados por participante e IC del 95 %

La probabilidad de cada participante se obtiene promediando las probabilidades ADHD de todas sus ventanas dentro de cada semilla. El umbral de decisión es 0.5. Los IC se calculan remuestreando participantes, manteniendo juntas sus diez predicciones; las semillas no se consideran observaciones clínicas independientes.

In [3]:
"""
Subject-level evaluation requested in Reviewer 1, Comments 1 and 6.

This cell reads only the previously exported participant predictions. It uses only out-of-fold
test predictions and treats the participant, rather than the random seed, as
the sampling unit for the 95% confidence intervals.
"""

from pathlib import Path
import shutil

import numpy as np
import pandas as pd
from scipy.stats import binomtest
from sklearn.metrics import roc_auc_score


SUBJECT_PROBABILITY_THRESHOLD = 0.5
N_SUBJECT_BOOTSTRAPS = 5000
SUBJECT_BOOTSTRAP_SEED = 20260826

SUBJECT_METRICS = (
    "accuracy",
    "sensitivity",
    "specificity",
    "precision",
    "f1_score",
    "roc_auc",
)

SUBJECT_METRIC_LABELS = {
    "accuracy": "Accuracy",
    "sensitivity": "Sensitivity",
    "specificity": "Specificity",
    "precision": "Precision",
    "f1_score": "F1-score",
    "roc_auc": "ROC-AUC",
}


def calculate_subject_metrics(y_true, y_prob, threshold=0.5):
    """Calculate the six metrics requested by the reviewer."""
    y_true = np.asarray(y_true, dtype=np.int64)
    y_prob = np.asarray(y_prob, dtype=np.float64)
    y_pred = (y_prob >= threshold).astype(np.int64)

    if y_true.shape != y_prob.shape:
        raise ValueError("y_true and y_prob must have the same shape.")
    if not np.isin(y_true, [0, 1]).all():
        raise ValueError("Subject labels must be binary (0=Control, 1=ADHD).")
    if not np.isfinite(y_prob).all():
        raise ValueError("Subject probabilities contain NaN or Inf values.")

    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    tn = int(np.sum((y_true == 0) & (y_pred == 0)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))

    def safe_divide(numerator, denominator):
        return float(numerator / denominator) if denominator else np.nan

    sensitivity = safe_divide(tp, tp + fn)
    specificity = safe_divide(tn, tn + fp)
    precision = safe_divide(tp, tp + fp)
    f1_score = safe_divide(
        2.0 * precision * sensitivity,
        precision + sensitivity,
    )

    try:
        roc_auc = float(roc_auc_score(y_true, y_prob))
    except ValueError:
        roc_auc = np.nan

    return {
        "accuracy": safe_divide(tp + tn, len(y_true)),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "f1_score": f1_score,
        "roc_auc": roc_auc,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def prepare_subject_predictions(subject_by_seed, expected_seeds=10):
    """
    Build one out-of-fold prediction per participant and seed.

    The participant probability is the arithmetic mean of all window-level
    ADHD probabilities belonging to that participant. The 0.5 threshold is
    applied only after this within-participant aggregation.
    """
    required_columns = {
        "model",
        "seed",
        "fold",
        "subject_id",
        "label",
        "class_name",
        "n_windows",
        "mean_prob_adhd",
    }
    missing = required_columns - set(subject_by_seed.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    subject_predictions = subject_by_seed[
        [
            "model",
            "seed",
            "fold",
            "subject_id",
            "label",
            "class_name",
            "n_windows",
            "mean_prob_adhd",
        ]
    ].copy()

    subject_predictions = subject_predictions.rename(
        columns={"mean_prob_adhd": "subject_prob_adhd"}
    )
    subject_predictions["subject_id"] = (
        subject_predictions["subject_id"].astype(str)
    )
    subject_predictions["label"] = (
        subject_predictions["label"].astype(int)
    )
    subject_predictions["seed"] = (
        subject_predictions["seed"].astype(int)
    )
    subject_predictions["fold"] = (
        subject_predictions["fold"].astype(int)
    )

    duplicated = subject_predictions.duplicated(
        subset=["model", "seed", "subject_id"],
        keep=False,
    )
    if duplicated.any():
        duplicate_rows = subject_predictions.loc[
            duplicated, ["model", "seed", "subject_id"]
        ]
        raise RuntimeError(
            "Expected exactly one row per participant and seed. Duplicates:\n"
            f"{duplicate_rows.to_string(index=False)}"
        )

    label_counts = subject_predictions.groupby("subject_id")["label"].nunique()
    if not (label_counts == 1).all():
        raise RuntimeError("At least one participant has inconsistent labels.")

    fold_counts = subject_predictions.groupby("subject_id")["fold"].nunique()
    if not (fold_counts == 1).all():
        raise RuntimeError(
            "At least one participant appears in more than one out-of-fold test fold."
        )

    seeds_per_subject = subject_predictions.groupby("subject_id")["seed"].nunique()
    if not (seeds_per_subject == expected_seeds).all():
        raise RuntimeError(
            "Every participant must have one out-of-fold prediction for each seed. "
            f"Observed seed counts: {seeds_per_subject.value_counts().to_dict()}"
        )

    n_subjects = subject_predictions["subject_id"].nunique()
    class_counts = (
        subject_predictions.drop_duplicates("subject_id")["label"].value_counts()
    )
    if n_subjects != 120 or class_counts.to_dict() != {0: 60, 1: 60}:
        raise RuntimeError(
            "Expected 120 participants (60 Control and 60 ADHD), but obtained "
            f"n={n_subjects}, classes={class_counts.to_dict()}."
        )

    subject_predictions["subject_pred"] = (
        subject_predictions["subject_prob_adhd"]
        >= SUBJECT_PROBABILITY_THRESHOLD
    ).astype(np.int64)
    subject_predictions["subject_correct"] = (
        subject_predictions["subject_pred"]
        == subject_predictions["label"]
    ).astype(np.int64)

    return subject_predictions.sort_values(
        ["seed", "fold", "subject_id"]
    ).reset_index(drop=True)


def metrics_for_each_seed(subject_predictions):
    """
    Descriptive model-initialization variability.

    These ten rows must not be used as ten independent clinical samples.
    """
    rows = []
    for seed, seed_frame in subject_predictions.groupby("seed", sort=True):
        seed_frame = seed_frame.sort_values("subject_id")
        metrics = calculate_subject_metrics(
            y_true=seed_frame["label"].to_numpy(),
            y_prob=seed_frame["subject_prob_adhd"].to_numpy(),
            threshold=SUBJECT_PROBABILITY_THRESHOLD,
        )
        rows.append(
            {
                "seed": int(seed),
                "n_subjects": int(len(seed_frame)),
                **metrics,
            }
        )
    return pd.DataFrame(rows)


def participant_stratified_bootstrap(
    subject_predictions,
    n_bootstrap=5000,
    random_state=20260826,
):
    """
    Estimate 95% CIs with participants as the resampling units.

    Within every bootstrap replicate, Control and ADHD participants are
    resampled separately with replacement. The same resampled participants are
    then applied to all ten seeds, metrics are calculated within each seed, and
    the ten metric values are averaged. Therefore, seeds are repeated model
    fits, not independent clinical observations.
    """
    probability_matrix = subject_predictions.pivot(
        index="subject_id",
        columns="seed",
        values="subject_prob_adhd",
    ).sort_index()

    labels = (
        subject_predictions.drop_duplicates("subject_id")
        .set_index("subject_id")["label"]
        .reindex(probability_matrix.index)
        .to_numpy(dtype=np.int64)
    )
    probabilities = probability_matrix.to_numpy(dtype=np.float64)
    seeds = probability_matrix.columns.to_numpy(dtype=int)

    if np.isnan(probabilities).any():
        raise RuntimeError("The participant-by-seed probability matrix is incomplete.")

    control_indices = np.flatnonzero(labels == 0)
    adhd_indices = np.flatnonzero(labels == 1)
    rng = np.random.default_rng(random_state)

    seed_metric_rows = []
    for seed_position, seed in enumerate(seeds):
        metrics = calculate_subject_metrics(
            labels,
            probabilities[:, seed_position],
            threshold=SUBJECT_PROBABILITY_THRESHOLD,
        )
        seed_metric_rows.append(
            {"seed": int(seed), **{name: metrics[name] for name in SUBJECT_METRICS}}
        )
    seed_metric_frame = pd.DataFrame(seed_metric_rows)

    point_estimates = {
        name: float(seed_metric_frame[name].mean())
        for name in SUBJECT_METRICS
    }

    bootstrap_values = np.empty(
        (n_bootstrap, len(SUBJECT_METRICS)),
        dtype=np.float64,
    )

    for bootstrap_index in range(n_bootstrap):
        sampled_indices = np.concatenate(
            [
                rng.choice(
                    control_indices,
                    size=len(control_indices),
                    replace=True,
                ),
                rng.choice(
                    adhd_indices,
                    size=len(adhd_indices),
                    replace=True,
                ),
            ]
        )
        sampled_labels = labels[sampled_indices]

        replicate_seed_metrics = []
        for seed_position in range(probabilities.shape[1]):
            metrics = calculate_subject_metrics(
                sampled_labels,
                probabilities[sampled_indices, seed_position],
                threshold=SUBJECT_PROBABILITY_THRESHOLD,
            )
            replicate_seed_metrics.append(
                [metrics[name] for name in SUBJECT_METRICS]
            )

        bootstrap_values[bootstrap_index] = np.nanmean(
            np.asarray(replicate_seed_metrics, dtype=np.float64),
            axis=0,
        )

    lower = np.nanpercentile(bootstrap_values, 2.5, axis=0)
    upper = np.nanpercentile(bootstrap_values, 97.5, axis=0)

    summary_rows = []
    for metric_position, metric in enumerate(SUBJECT_METRICS):
        estimate = point_estimates[metric]
        ci_lower = float(lower[metric_position])
        ci_upper = float(upper[metric_position])
        summary_rows.append(
            {
                "metric": metric,
                "metric_label": SUBJECT_METRIC_LABELS[metric],
                "estimate": estimate,
                "ci_95_lower": ci_lower,
                "ci_95_upper": ci_upper,
                "estimate_percent": 100.0 * estimate,
                "ci_95_lower_percent": 100.0 * ci_lower,
                "ci_95_upper_percent": 100.0 * ci_upper,
                "n_subjects": int(len(labels)),
                "n_control": int(len(control_indices)),
                "n_adhd": int(len(adhd_indices)),
                "n_seeds": int(len(seeds)),
                "n_bootstrap": int(n_bootstrap),
                "sampling_unit": "participant",
                "bootstrap_type": "stratified percentile bootstrap",
            }
        )

    bootstrap_frame = pd.DataFrame(
        bootstrap_values,
        columns=SUBJECT_METRICS,
    )
    bootstrap_frame.insert(
        0,
        "bootstrap_replicate",
        np.arange(1, n_bootstrap + 1),
    )

    return pd.DataFrame(summary_rows), bootstrap_frame, seed_metric_frame


def build_consensus_subject_predictions(subject_predictions):
    """Create one seed-averaged probability per participant for inspection."""
    consensus = (
        subject_predictions.groupby(
            ["model", "subject_id", "label", "class_name"],
            as_index=False,
        )
        .agg(
            fold=("fold", "first"),
            n_seeds=("seed", "nunique"),
            n_windows=("n_windows", "first"),
            mean_prob_adhd_across_seeds=("subject_prob_adhd", "mean"),
            std_prob_adhd_across_seeds=("subject_prob_adhd", "std"),
        )
    )
    consensus["consensus_pred"] = (
        consensus["mean_prob_adhd_across_seeds"]
        >= SUBJECT_PROBABILITY_THRESHOLD
    ).astype(np.int64)
    consensus["consensus_correct"] = (
        consensus["consensus_pred"] == consensus["label"]
    ).astype(np.int64)
    return consensus.sort_values(["label", "subject_id"]).reset_index(drop=True)


def exact_mcnemar_subject_level(model_a, model_b, name_a="model_a", name_b="model_b"):
    """
    Exact paired McNemar test for two models' consensus subject decisions.

    Each input must contain subject_id, label, and consensus_pred. This helper
    is ready for use after equivalent out-of-fold predictions are exported for
    each baseline. Holm correction should then be applied across comparisons.
    """
    required = {"subject_id", "label", "consensus_pred"}
    for name, frame in ((name_a, model_a), (name_b, model_b)):
        missing = required - set(frame.columns)
        if missing:
            raise KeyError(f"{name} is missing columns: {sorted(missing)}")

    paired = model_a[list(required)].merge(
        model_b[list(required)],
        on="subject_id",
        suffixes=(f"_{name_a}", f"_{name_b}"),
        validate="one_to_one",
    )
    if not np.array_equal(
        paired[f"label_{name_a}"].to_numpy(),
        paired[f"label_{name_b}"].to_numpy(),
    ):
        raise RuntimeError("The paired models contain inconsistent labels.")

    labels = paired[f"label_{name_a}"].to_numpy(dtype=int)
    correct_a = paired[f"consensus_pred_{name_a}"].to_numpy(dtype=int) == labels
    correct_b = paired[f"consensus_pred_{name_b}"].to_numpy(dtype=int) == labels
    a_only = int(np.sum(correct_a & ~correct_b))
    b_only = int(np.sum(~correct_a & correct_b))
    discordant = a_only + b_only
    p_value = (
        float(binomtest(a_only, discordant, p=0.5).pvalue)
        if discordant > 0
        else 1.0
    )
    return {
        "model_a": name_a,
        "model_b": name_b,
        "n_subjects": int(len(paired)),
        "a_correct_b_incorrect": a_only,
        "a_incorrect_b_correct": b_only,
        "exact_mcnemar_p": p_value,
    }


def run_subject_level_review_analysis(
    input_path,
    output_root,
    n_bootstrap=N_SUBJECT_BOOTSTRAPS,
    random_state=SUBJECT_BOOTSTRAP_SEED,
):
    input_path = Path(input_path)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)
    if not input_path.exists():
        raise FileNotFoundError(
            "No se encontró el resumen por participante y semilla:\n"
            f"{input_path}"
        )

    subject_by_seed = pd.read_csv(input_path)
    subject_predictions = prepare_subject_predictions(subject_by_seed)
    descriptive_seed_metrics = metrics_for_each_seed(subject_predictions)
    metric_summary, bootstrap_distribution, bootstrap_seed_metrics = (
        participant_stratified_bootstrap(
            subject_predictions,
            n_bootstrap=n_bootstrap,
            random_state=random_state,
        )
    )
    consensus_predictions = build_consensus_subject_predictions(
        subject_predictions
    )

    manuscript_table = metric_summary[
        [
            "metric_label",
            "estimate_percent",
            "ci_95_lower_percent",
            "ci_95_upper_percent",
        ]
    ].copy()
    manuscript_table["Result (95\\% CI), \\%"] = manuscript_table.apply(
        lambda row: (
            f"{row['estimate_percent']:.1f} "
            f"({row['ci_95_lower_percent']:.1f}--"
            f"{row['ci_95_upper_percent']:.1f})"
        ),
        axis=1,
    )
    manuscript_table = manuscript_table[
        ["metric_label", "Result (95\\% CI), \\%"]
    ].rename(columns={"metric_label": "Metric"})

    output_paths = {
        "subject_predictions_by_seed": (
            output_root / "MultiStream_subject_level_predictions_by_seed.csv"
        ),
        "subject_metrics_by_seed": (
            output_root / "MultiStream_subject_level_metrics_by_seed.csv"
        ),
        "subject_metrics_95ci": (
            output_root / "MultiStream_subject_level_metrics_95CI.csv"
        ),
        "bootstrap_distribution": (
            output_root / "MultiStream_subject_level_bootstrap_distribution.csv"
        ),
        "bootstrap_seed_metrics": (
            output_root / "MultiStream_subject_level_point_metrics_by_seed.csv"
        ),
        "consensus_predictions": (
            output_root / "MultiStream_subject_level_consensus_predictions.csv"
        ),
        "manuscript_table_csv": (
            output_root / "MultiStream_subject_level_table_for_manuscript.csv"
        ),
        "manuscript_table_latex": (
            output_root / "MultiStream_subject_level_table_for_manuscript.tex"
        ),
    }

    subject_predictions.to_csv(
        output_paths["subject_predictions_by_seed"], index=False
    )
    descriptive_seed_metrics.to_csv(
        output_paths["subject_metrics_by_seed"], index=False
    )
    metric_summary.to_csv(output_paths["subject_metrics_95ci"], index=False)
    bootstrap_distribution.to_csv(
        output_paths["bootstrap_distribution"], index=False
    )
    bootstrap_seed_metrics.to_csv(
        output_paths["bootstrap_seed_metrics"], index=False
    )
    consensus_predictions.to_csv(
        output_paths["consensus_predictions"], index=False
    )
    manuscript_table.to_csv(
        output_paths["manuscript_table_csv"], index=False
    )
    latex_lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        (
            r"\caption{Participant-level classification performance of "
            r"MultiStream. Values are averaged across ten random training seeds; "
            r"95\% confidence intervals were estimated using 5,000 "
            r"stratified bootstrap resamples at the participant level.}"
        ),
        r"\label{tab:subject_level_performance}",
        r"\begin{tabular}{lc}",
        r"\hline",
        r"Metric & Result (95\% CI), \% \\",
        r"\hline",
    ]
    for _, row in manuscript_table.iterrows():
        latex_lines.append(
            f"{row['Metric']} & {row['Result (95\\% CI), \\%']} \\\\"
        )
    latex_lines.extend(
        [
            r"\hline",
            r"\end{tabular}",
            r"\end{table}",
        ]
    )
    output_paths["manuscript_table_latex"].write_text(
        "\n".join(latex_lines) + "\n",
        encoding="utf-8",
    )

    print("\n" + "=" * 80)
    print("REVIEWER 1 - COMMENTS 1 AND 6: SUBJECT-LEVEL RESULTS")
    print("=" * 80)
    print(
        "Aggregation: arithmetic mean of window probabilities within each "
        "participant and seed; threshold = 0.5."
    )
    print(
        "Inference: 95% stratified percentile bootstrap with the participant "
        "as the resampling unit."
    )
    print(
        "The ten seeds quantify model-initialization variability and are not "
        "treated as independent clinical observations.\n"
    )
    print(manuscript_table.to_string(index=False))
    print("\nFiles saved in:", output_root)

    return {
        "subject_predictions_by_seed": subject_predictions,
        "subject_metrics_by_seed": descriptive_seed_metrics,
        "subject_metrics_95ci": metric_summary,
        "bootstrap_distribution": bootstrap_distribution,
        "consensus_predictions": consensus_predictions,
        "manuscript_table": manuscript_table,
        "output_paths": output_paths,
    }


subject_level_results = run_subject_level_review_analysis(
    input_path=SUBJECT_SUMMARY_PATH,
    output_root=OUTPUT_ROOT,
    n_bootstrap=N_SUBJECT_BOOTSTRAPS,
    random_state=SUBJECT_BOOTSTRAP_SEED,
)

# Paquete único para descargar desde Kaggle. Incluye métricas, predicciones,
# IC 95%, tabla para el manuscrito y consenso; no contiene modelos.
archive_path = shutil.make_archive(
    "/kaggle/working/MultiStream_analysis_only_results",
    "zip",
    root_dir=OUTPUT_ROOT,
)
print("Archivo ZIP final:", archive_path)



REVIEWER 1 - COMMENTS 1 AND 6: SUBJECT-LEVEL RESULTS
Aggregation: arithmetic mean of window probabilities within each participant and seed; threshold = 0.5.
Inference: 95% stratified percentile bootstrap with the participant as the resampling unit.
The ten seeds quantify model-initialization variability and are not treated as independent clinical observations.

     Metric Result (95\% CI), \%
   Accuracy    56.2 (51.2--61.4)
Sensitivity    93.7 (88.3--98.0)
Specificity    18.8 (10.2--28.3)
  Precision    53.6 (50.6--56.8)
   F1-score    68.2 (64.8--71.4)
    ROC-AUC    60.3 (50.9--69.6)

Files saved in: /kaggle/working/MultiStream_analysis_only
Archivo ZIP final: /kaggle/working/MultiStream_analysis_only_results.zip
